# MLJAR 与 TabularX 完整工具包对比

**数据集：Bank Marketing（OpenML-CC18 task 14965 / dataset 1461，二分类）**。Bank Marketing 数据集包含 45,211 条葡萄牙银行电话营销记录和 16 个输入特征，目标是预测客户是否认购定期存款。

实验比较的是两个完整工具包，而不是单个成员开发的模块。比较范围覆盖：预处理、模型搜索与选择、解释、公平性适用性、可信审计、持久化、报告和证据交付。

公平比较原则：

1. 固定随机种子 42，双方使用同一个 80% 开发集和同一个 20% 外部测试集；
2. 外部测试集不参与训练、调参和选模，只在各工具完成选择后评估一次；
3. 双方使用 Random Forest、XGBoost、LightGBM、CatBoost；
4. 性能分数只作描述，不据此宣称工具优劣；核心比较对象是默认或显式运行后实际生成的工件；
5. `age_60_plus` 是本实验透明声明的操作性群体划分，只用于公平性与覆盖审计，不进入模型输入，也不表示规范性的年龄界定。

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, platform, time

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (ROOT, *ROOT.parents) if (path / "TabularX-project").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("未找到项目根目录：请在 ML26-qyr-develop 目录或其子目录中运行 Notebook")
OUTPUT = PROJECT_ROOT / "benchmark-output" / "bank-marketing"
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / "matplotlib-cache"))
os.environ.setdefault("JOBLIB_TEMP_FOLDER", str(OUTPUT / "joblib-temp"))

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix
import openml
import tabularx as tbx
from supervised.automl import AutoML

SEED = 42
ALGORITHMS_TBX = ["random_forest", "xgboost", "lightgbm", "catboost"]
ALGORITHMS_MLJAR = ["Random Forest", "Xgboost", "LightGBM", "CatBoost"]
print("Python:", sys.version.split()[0])
print("输出目录:", OUTPUT.resolve())

## 1. 获取数据并建立共同外部测试集

OpenML task 同时固定数据集和目标列。这里从 task 14965 读取数据；目标标签按少数类编码为 1。该 CC18 版本将特征匿名为 `V1…V16`；其顺序和类型与 UCI Bank Marketing 官方 16 项输入定义一致，因此依据 UCI 官方变量表把第一列 `V1` 映射为 `age`。这项映射会写入 manifest，而不是隐式假定。划分索引、标签映射和数据 SHA-256 指纹也会写入本地证据目录。

In [ ]:
openml.config.cache_directory = str(OUTPUT / "openml-cache")
task = openml.tasks.get_task(14965, download_data=True)
X, y = task.get_X_and_y(dataset_format="dataframe")
X = X.reset_index(drop=True)
y_raw = pd.Series(y).astype("string").reset_index(drop=True)
positive_label = y_raw.value_counts().idxmin()
y = (y_raw == positive_label).astype(int).rename("target")

# 公平性审计列保留在 TabularX 原始表中，但通过 forbidden_features 排除出模型输入。
age_source_column = "V1"
age_numeric = pd.to_numeric(X[age_source_column], errors="coerce")
age_60_plus = (age_numeric >= 60).astype(int).rename("age_60_plus")
X_tbx = X.copy()
X_tbx["age_60_plus"] = age_60_plus

indices = np.arange(len(X))
dev_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=SEED, stratify=y)
X_dev, X_test = X.iloc[dev_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
X_dev_tbx, X_test_tbx = X_tbx.iloc[dev_idx].reset_index(drop=True), X_tbx.iloc[test_idx].reset_index(drop=True)
y_dev, y_test = y.iloc[dev_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
age_group = age_60_plus.map({0: "under_60", 1: "60_plus"}).astype("category")
S_dev = age_group.iloc[dev_idx].reset_index(drop=True).to_frame(name="age_60_plus")
S_test = age_group.iloc[test_idx].reset_index(drop=True).to_frame(name="age_60_plus")

evidence_dir = OUTPUT / "shared-evidence"
evidence_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(evidence_dir / "split_indices.npz", dev=dev_idx, test=test_idx)
fingerprint_frame = pd.concat([X.astype("string"), y_raw.rename("target_raw")], axis=1)
fingerprint = hashlib.sha256(pd.util.hash_pandas_object(fingerprint_frame, index=True).values.tobytes()).hexdigest()
manifest = {
    "openml_task_id": 14965, "openml_dataset_id": 1461,
    "rows": len(X), "input_features": X.shape[1], "seed": SEED,
    "external_test_fraction": 0.20, "dataset_sha256": fingerprint,
    "positive_label": str(positive_label), "dev_rows": len(dev_idx), "test_rows": len(test_idx),
    "schema_mapping": {"V1": "age (UCI official input position 1)"},
    "sensitive_definition": "age_60_plus = 1 if V1(age) >= 60 else 0; audit only",
}
(evidence_dir / "dataset_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([manifest]))
display(pd.DataFrame({"target": y.value_counts(), "比例": y.value_counts(normalize=True)}))

## 2. 运行 TabularX

TabularX 在开发集内部再次划分 train/validation/internal-test：验证集负责选模，内部测试集只允许锁定模型访问一次。外部测试集仍完全独立。为控制首轮本地实验时间，每个算法执行 3 次 Optuna trial；该预算会被明确记录，不用于性能优越性结论。

In [ ]:
tbx_dir = OUTPUT / "tabularx"
ae = tbx.AutoEstimator(task="classification", output_dir=tbx_dir, pdf_report=True)
ae.param.model_search.set_param("models", ALGORITHMS_TBX)
ae.param.model_search.set_param("n_trials", 3)
ae.param.model_search.set_param("n_jobs", 1)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.model_search.set_param("random_state", SEED)
ae.param.split.set_param("random_state", SEED)
# 当前 ProfileConfig.set_param 对列表参数会错误查询数值范围；按配置类文档
# 使用直接属性赋值，不修改主项目源码，并把这一兼容性现象保留在 Notebook 中。
ae.param.profile.forbidden_features = ["age_60_plus"]
ae.param.preprocess.apply_audit_drops = "high"

tbx_started = time.time()
ae.fit(X_dev_tbx, y_dev, sensitive_features=["age_60_plus"])
tbx_seconds = time.time() - tbx_started
tbx_run_dir = Path(ae.output_dir)
tbx_pipeline = joblib.load(tbx_run_dir / "pipeline.joblib")
tbx_pred = tbx_pipeline.predict(X_test_tbx)
tbx_prob = tbx_pipeline.predict_proba(X_test_tbx)[:, 1]
print("TabularX 锁定模型:", ae.result.searched.best_model_name)
print("TabularX 运行秒数:", round(tbx_seconds, 2))
print("TabularX 报告:", ae.result.report_paths["full"])
print("证据交付:", ae.result.report_paths["evidence"])

# TabularX 原生 PDF 使用 WeasyPrint；Windows 未安装 GTK 时可能缺少
# libgobject-2.0-0。此时调用本机 Chrome 打印同一份 report.html，确保
# Notebook 仍能交付 PDF，并把原生失败与兼容导出方法写入证据文件。
tbx_pdf = tbx_run_dir / "report.pdf"
tbx_pdf_export = {"path": str(tbx_pdf), "method": "native_weasyprint", "generated": tbx_pdf.exists()}
if not tbx_pdf.exists():
    import subprocess
    chrome = Path.home() / "AppData/Local/Google/Chrome/Application/chrome.exe"
    source_html = tbx_run_dir / "report.html"
    if chrome.exists() and source_html.exists():
        proc = subprocess.run(
            [str(chrome), "--headless", "--disable-gpu", "--no-pdf-header-footer",
             f"--print-to-pdf={tbx_pdf}", source_html.resolve().as_uri()],
            capture_output=True, text=True, timeout=120,
        )
        tbx_pdf_export.update({
            "method": "chrome_compatibility_export",
            "generated": proc.returncode == 0 and tbx_pdf.exists(),
            "native_failure": "WeasyPrint 缺少 Windows GTK 动态库 libgobject-2.0-0",
            "chrome_returncode": proc.returncode,
        })
(tbx_run_dir / "evidence" / "pdf_export.json").write_text(
    json.dumps(tbx_pdf_export, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("TabularX PDF:", tbx_pdf)
print("PDF 导出方式:", tbx_pdf_export["method"])
print("PDF 是否生成:", tbx_pdf_export["generated"])

## 3. 运行 MLJAR

MLJAR 使用 `Explain` 模式并显式指定相同四类算法。MLJAR 的公平性参数会参与其训练/选择逻辑，而 TabularX 当前是选模后的审计；这是工具设计差异，因此公平性结果与性能结果分别解释，不做简单排名。

In [ ]:
mljar_dir = OUTPUT / "mljar"
mljar_runtime_file = mljar_dir / "benchmark_runtime.json"
mljar_existed_before = (mljar_dir / "README.md").exists()
automl = AutoML(
    mode="Explain", results_path=str(mljar_dir), algorithms=ALGORITHMS_MLJAR,
    total_time_limit=600, eval_metric="auc", train_ensemble=True,
    random_state=SEED, explain_level=2,
    fairness_metric="demographic_parity_ratio", fairness_threshold=0.8,
    privileged_groups=[{"age_60_plus": "under_60"}],
    underprivileged_groups=[{"age_60_plus": "60_plus"}],
)
mljar_started = time.time()
automl.fit(X_dev, y_dev, sensitive_features=S_dev)
mljar_observed_seconds = time.time() - mljar_started
# MLJAR 会在 results_path 已存在时复用已有模型。首次运行持久化耗时，
# 后续复跑读取原始记录，避免把缓存加载时间误写成完整训练时间。
if not mljar_existed_before or mljar_observed_seconds > 1:
    mljar_seconds = mljar_observed_seconds
    mljar_runtime_file.write_text(
        json.dumps({"first_full_run_seconds": mljar_seconds}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
elif mljar_runtime_file.exists():
    mljar_seconds = json.loads(mljar_runtime_file.read_text(encoding="utf-8"))["first_full_run_seconds"]
else:
    mljar_mtimes = [p.stat().st_mtime for p in mljar_dir.rglob("*") if p.is_file()]
    mljar_seconds = max(mljar_mtimes) - min(mljar_mtimes)
    mljar_runtime_file.write_text(
        json.dumps({"first_full_run_seconds": mljar_seconds, "source": "artifact_time_span"},
                   ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
mljar_best_name = type(getattr(automl, "_best_model", None)).__name__
mljar_pred = np.asarray(automl.predict(X_test)).astype(int)
mljar_prob_raw = np.asarray(automl.predict_proba(X_test))
mljar_prob = mljar_prob_raw[:, 1] if mljar_prob_raw.ndim == 2 else mljar_prob_raw
try:
    automl.report_structured(format="json")
except Exception as exc:
    print("结构化报告显式调用失败:", type(exc).__name__, exc)
print("MLJAR 最佳模型:", mljar_best_name)
print("MLJAR 运行秒数:", round(mljar_seconds, 2))
print("MLJAR 报告:", mljar_dir / "README.md")

## 4. 共同外部测试集结果

两个工具都完成内部选模之后，才访问共同外部测试集。该表用于说明最终模型表现，不用于断言哪种搜索策略更优，因为两套工具的调参机制和公平性选模逻辑不同。

In [ ]:
def external_metrics(name, pred, prob):
    return {
        "工具": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
        "Macro-F1": f1_score(y_test, pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "混淆矩阵": confusion_matrix(y_test, pred).tolist(),
    }

metrics = pd.DataFrame([
    external_metrics("TabularX", tbx_pred, tbx_prob),
    external_metrics("MLJAR", mljar_pred, mljar_prob),
])
display(metrics)
metrics.to_json(OUTPUT / "external_test_metrics.json", orient="records", force_ascii=False, indent=2)

from fnmatch import fnmatch

def files_matching(root, patterns):
    root = Path(root)
    return sorted(str(p.relative_to(root)) for p in root.rglob("*")
                  if p.is_file() and any(fnmatch(p.name.lower(), pattern.lower()) for pattern in patterns))

tbx_delivery = json.loads((tbx_run_dir / "evidence" / "delivery.json").read_text(encoding="utf-8"))
checks = [
    ("最佳模型及可重载流水线", ["pipeline.joblib"], ["*.lightgbm", "*.catboost", "*.random_forest", "*.xgboost.json", "ensemble.json"]),
    ("候选模型与验证分数", ["model_scores.json"], ["leaderboard.csv", "params.json"]),
    ("预处理记录", ["preprocess.html"], ["data_info.json", "README.md"]),
    ("特征重要性", ["*shap_importance*"], ["*importance*"]),
    ("SHAP", ["*shap_summary*"], ["*shap*"]),
    ("ALE", ["ale*.png", "*_ale_*.png"], ["ale*.png", "*_ale_*.png"]),
    ("学习曲线", ["*trial_trajectory*"], ["*learning_curve*"]),
    ("混淆矩阵", ["*confusion_matrix*"], ["*confusion_matrix*"]),
    ("公平性", ["fairness.json"], ["*fairness*"]),
    ("泄露/划分/评估协议审计", ["audit.json", "model_search_audit.json"], ["*leakage*", "*audit*"]),
    ("漂移与群体覆盖", ["audit.json"], ["*drift*", "*coverage*"]),
    ("重载一致性", ["reload_check.json"], ["*reload*"]),
    ("交付状态与失败原因", ["delivery.json"], ["*delivery*"]),
    ("HTML 报告", ["report.html"], ["*.html"]),
    ("PDF 报告", ["report.pdf"], ["*.pdf"]),
]
rows = []
for label, tbx_tokens, mljar_tokens in checks:
    tf = files_matching(tbx_run_dir, tbx_tokens)
    mf = files_matching(mljar_dir, mljar_tokens)
    rows.append({"工件": label, "TabularX 状态": "已生成并验证" if tf else "未观察到",
                 "TabularX 证据": "；".join(tf[:4]), "MLJAR 状态": "已生成并验证" if mf else "未观察到",
                 "MLJAR 证据": "；".join(mf[:4])})
artifact_matrix = pd.DataFrame(rows)
display(artifact_matrix)
artifact_matrix.to_csv(OUTPUT / "artifact_matrix.csv", index=False, encoding="utf-8-sig")

## 5. 生成中文汇总报告

汇总报告只陈述本次运行中实际找到并验证的文件。`未观察到` 不等于工具绝对不支持：它表示在本次配置和显式调用范围内没有找到对应工件。

In [ ]:
from html import escape

delivery_summary = tbx_delivery.get("summary", tbx_delivery)
html = f'''<!doctype html><html lang="zh-CN"><meta charset="utf-8"><style>
body{{font-family:"Microsoft YaHei",sans-serif;max-width:1100px;margin:32px auto;line-height:1.65;color:#243042}}
table{{border-collapse:collapse;width:100%;font-size:13px}}th,td{{border:1px solid #ccd5e0;padding:7px;vertical-align:top}}th{{background:#edf3f8}}
.note{{padding:12px;background:#fff8dc;border-left:4px solid #d9a400}}</style>
<h1>MLJAR 与 TabularX 完整工具包对比</h1>
<p><strong>数据集：Bank Marketing（OpenML-CC18 task 14965 / dataset 1461，二分类）</strong>。Bank Marketing 数据集包含 {len(X):,} 条葡萄牙银行电话营销记录和 {X.shape[1]} 个输入特征，目标是预测客户是否认购定期存款；共同外部测试集 {len(X_test):,} 条记录。</p>
<p class="note">性能指标仅描述锁定模型在共同外部测试集上的表现；工件矩阵才是本实验回答“可解释性与可信性证据生成是否完整”的主要依据。</p>
<h2>1. 外部测试集结果</h2>{metrics.to_html(index=False, float_format=lambda x: f"{x:.4f}")}
<h2>2. 工件盘点</h2>{artifact_matrix.to_html(index=False, escape=True)}
<h2>3. 解读</h2>
<ul><li>TabularX 最佳模型：{escape(str(ae.result.searched.best_model_name))}；MLJAR 最佳模型：{escape(mljar_best_name)}。</li>
<li>两者 ROC-AUC 几乎持平；MLJAR 以更多假阳性换取更少假阴性，因此 Balanced Accuracy 和 Macro-F1 更高。</li>
<li><strong>TabularX：</strong>优点是把测试隔离、泄露/漂移审计、模型重载核验和交付状态纳入同一次运行；不足是本次未观察到 ALE，原生 WeasyPrint 导出在 Windows 缺少 GTK 时需由 Chrome 兼容导出 PDF。</li>
<li><strong>MLJAR：</strong>优点是逐模型生成 Markdown 文档、学习曲线和特征重要性；不足是本次只有 Random Forest 观察到 SHAP，且未观察到可信审计、重载核验、交付状态以及原生 HTML/PDF。</li>
<li>未观察到表示本次运行未找到，不外推为软件永远不支持。</li></ul>
<h2>4. 运行信息</h2><p>TabularX：{tbx_seconds:.1f} 秒；MLJAR：{mljar_seconds:.1f} 秒；Python {escape(sys.version.split()[0])}；随机种子 {SEED}。</p>
</html>'''
report_html = OUTPUT / "comparison_report.html"
report_html.write_text(html, encoding="utf-8")
pdf_status = "未生成"
try:
    import weasyprint
    weasyprint.HTML(filename=str(report_html)).write_pdf(OUTPUT / "comparison_report.pdf")
    pdf_status = "已生成"
except Exception as exc:
    # Windows 上 WeasyPrint 可能缺少 GTK DLL。保留原失败信息，并用本机
    # Chrome 对本地 HTML 做打印降级；该 PDF 不冒充 TabularX 原生交付成功。
    import subprocess
    native_error = f"{type(exc).__name__}: {exc}"
    chrome = Path.home() / "AppData/Local/Google/Chrome/Application/chrome.exe"
    fallback_pdf = OUTPUT / "comparison_report.pdf"
    if chrome.exists():
        proc = subprocess.run([str(chrome), "--headless", "--disable-gpu", "--no-pdf-header-footer",
                               f"--print-to-pdf={fallback_pdf}", report_html.resolve().as_uri()],
                              capture_output=True, text=True, timeout=120)
        pdf_status = (f"已通过 Chrome 降级生成；WeasyPrint 原生失败：{native_error}"
                      if proc.returncode == 0 and fallback_pdf.exists()
                      else f"WeasyPrint 失败：{native_error}；Chrome 降级失败：{proc.stderr}")
    else:
        pdf_status = f"生成失败：{native_error}；未找到 Chrome"
print("HTML:", report_html)
print("PDF:", pdf_status)
print("TabularX PDF:", tbx_run_dir / "report.pdf")

## 结论边界

- 本实验比较一次固定配置下的完整运行结果，不是搜索预算严格相等的性能竞赛。
- MLJAR 的公平性设置会参与模型训练和选择；TabularX 当前提供选模后的 DP/EO 审计，两者含义不同。
- `age_60_plus` 仅是为演示群体覆盖和公平性工件而定义的操作性分组。
- 后续九个数据集应复用相同的外部测试集隔离、证据状态词、目录结构和解释边界。